In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.tree import plot_tree

In [ ]:
A_Data = pd.read_csv("CancerData.csv")

systemic_features = [
    "Fatigue",
    "Weight Loss",
    "Swallowing Difficulty",
    "Clubbing of Finger Nails",
    "Snoring"
]

A_Data = A_Data.drop_duplicates().copy()

X = A_Data[systemic_features]
Y = A_Data["Level"]

assert "Level" not in X.columns, "Target leakage: Level is inside X"

print("\nRisk level labels:")
print(sorted(Y.unique()))

display(A_Data[["Patient Id"] + systemic_features + ["Level"]].head())

In [ ]:
systemic_data = A_Data[systemic_features + ["Level"]].copy()

print("Missing values in selected data:")
display(systemic_data.isnull().sum())

print("Risk level counts:")
display(systemic_data["Level"].value_counts())

print("Mean systemic symptom score by risk level:")
symptom_means = systemic_data.groupby("Level")[systemic_features].mean().reindex(["Low", "Medium", "High"])
display(symptom_means.round(2))

In [ ]:
X_train, X_secondary, Y_train, Y_secondary = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=0,
    stratify=Y
)

X_validation, X_test, Y_validation, Y_test = train_test_split(
    X_secondary,
    Y_secondary,
    test_size=0.5,
    random_state=0,
    stratify=Y_secondary
)

split_summary = pd.DataFrame({
    "Split": ["Training", "Validation", "Test"],
    "Rows": [len(X_train), len(X_validation), len(X_test)]
})

display(split_summary)

print("Training class counts:")
display(Y_train.value_counts())

print("Validation class counts:")
display(Y_validation.value_counts())

print("Test class counts:")
display(Y_test.value_counts())

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

baseline_model = DummyClassifier(strategy="most_frequent")
baseline_model.fit(X_train, Y_train)

baseline_validation_pred = baseline_model.predict(X_validation)

baseline_cv_scores = cross_val_score(
    baseline_model,
    X_train,
    Y_train,
    cv=cv,
    scoring="recall_macro"
)

baseline_results = pd.DataFrame([{
    "Model": "Most Frequent Baseline",
    "Validation Accuracy": accuracy_score(Y_validation, baseline_validation_pred),
    "Validation Balanced Accuracy": balanced_accuracy_score(Y_validation, baseline_validation_pred),
    "Validation Macro Recall": recall_score(
        Y_validation,
        baseline_validation_pred,
        average="macro",
        zero_division=0
    ),
    "Cross-Validation Macro Recall": baseline_cv_scores.mean()
}]).round(3)

display(baseline_results)

def get_class_recall(y_true, y_pred, class_label="High"):
    report = classification_report(
        y_true,
        y_pred,
        output_dict=True,
        zero_division=0
    )
    if class_label in report:
        return report[class_label]["recall"]
    return np.nan

def evaluate_random_forest(params):
    model = RandomForestClassifier(
        **params,
        random_state=0,
        class_weight="balanced"
    )

    model.fit(X_train, Y_train)

    train_pred = model.predict(X_train)
    validation_pred = model.predict(X_validation)

    report = classification_report(
        Y_validation,
        validation_pred,
        output_dict=True,
        zero_division=0
    )

    return {
        "Training Accuracy": accuracy_score(Y_train, train_pred),
        "Validation Accuracy": accuracy_score(Y_validation, validation_pred),
        "Validation Balanced Accuracy": balanced_accuracy_score(Y_validation, validation_pred),
        "Validation Macro Recall": recall_score(
            Y_validation,
            validation_pred,
            average="macro",
            zero_division=0
        ),
        "Validation Macro F1": f1_score(
            Y_validation,
            validation_pred,
            average="macro",
            zero_division=0
        ),
        "Validation High-Risk Recall": report["High"]["recall"] if "High" in report else np.nan,
        "Train-Val Gap": accuracy_score(Y_train, train_pred) - accuracy_score(Y_validation, validation_pred)
    }

In [ ]:
base_params = {
    "n_estimators": 30,
    "max_depth": 1,
    "min_samples_split": 2,
    "min_impurity_decrease": 0.0
}

In [ ]:
max_depth_results = []

for max_depth in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    params = base_params.copy()
    params["max_depth"] = max_depth

    scores = evaluate_random_forest(params)

    max_depth_results.append({
        "max_depth": max_depth,
        **scores
    })

max_depth_results = pd.DataFrame(max_depth_results).round(3)
display(max_depth_results)

plt.figure(figsize=(7, 4))
plt.plot(
    max_depth_results["max_depth"],
    max_depth_results["Training Accuracy"],
    label="Training Data",
    color="#8ECAE6"
)
plt.plot(
    max_depth_results["max_depth"],
    max_depth_results["Validation Accuracy"],
    label="Validation Data",
    color="#023047"
)
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("max_depth")
plt.legend()
plt.tight_layout()
plt.savefig("max_depth.png", dpi=300, bbox_inches="tight")
plt.show()

best_score = max_depth_results["Validation Balanced Accuracy"].max()

candidates = max_depth_results[
    max_depth_results["Validation Balanced Accuracy"] >= best_score - 0.02
].copy()

best_max_depth = candidates.sort_values("max_depth").iloc[0]["max_depth"]

print("Selected max_depth:", best_max_depth)

base_params["max_depth"] = int(best_max_depth)
print("Updated parameters:")
print(base_params)

In [ ]:
min_samples_split_results = []

for min_samples_split in [2, 5, 10, 20, 30, 50]:
    params = base_params.copy()
    params["min_samples_split"] = min_samples_split

    scores = evaluate_random_forest(params)

    min_samples_split_results.append({
        "min_samples_split": min_samples_split,
        **scores
    })

min_samples_split_results = pd.DataFrame(min_samples_split_results).round(3)
display(min_samples_split_results)

plt.figure(figsize=(7, 4))
plt.plot(
    min_samples_split_results["min_samples_split"],
    min_samples_split_results["Training Accuracy"],
    label="Training Data",
    color="#8ECAE6"
)
plt.plot(
    min_samples_split_results["min_samples_split"],
    min_samples_split_results["Validation Accuracy"],
    label="Validation Data",
    color="#023047"
)
plt.xlabel("min_samples_split")
plt.ylabel("Accuracy")
plt.title("min_samples_split")
plt.legend()
plt.tight_layout()
plt.savefig("min_samples_split.png", dpi=300, bbox_inches="tight")
plt.show()

best_score = min_samples_split_results["Validation Balanced Accuracy"].max()

candidates = min_samples_split_results[
    min_samples_split_results["Validation Balanced Accuracy"] >= best_score - 0.02
].copy()

best_min_samples_split = candidates.sort_values(
    "min_samples_split",
    ascending=False
).iloc[0]["min_samples_split"]

print("Selected min_samples_split:", best_min_samples_split)

base_params["min_samples_split"] = int(best_min_samples_split)
print("Updated parameters:")
print(base_params)

In [ ]:
min_impurity_results = []

for min_impurity_decrease in [0.0, 0.001, 0.005, 0.01, 0.02, 0.05]:
    params = base_params.copy()
    params["min_impurity_decrease"] = min_impurity_decrease

    scores = evaluate_random_forest(params)

    min_impurity_results.append({
        "min_impurity_decrease": min_impurity_decrease,
        **scores
    })

min_impurity_results = pd.DataFrame(min_impurity_results).round(3)
display(min_impurity_results)

plt.figure(figsize=(7, 4))
plt.plot(
    min_impurity_results["min_impurity_decrease"],
    min_impurity_results["Training Accuracy"],
    label="Training Data",
    color="#8ECAE6"
)
plt.plot(
    min_impurity_results["min_impurity_decrease"],
    min_impurity_results["Validation Accuracy"],
    label="Validation Data",
    color="#023047"
)
plt.xlabel("min_impurity_decrease")
plt.ylabel("Accuracy")
plt.title("min_impurity_decrease")
plt.legend()
plt.tight_layout()
plt.savefig("min_impurity_decrease.png", dpi=300, bbox_inches="tight")
plt.show()

best_score = min_impurity_results["Validation Balanced Accuracy"].max()

candidates = min_impurity_results[
    min_impurity_results["Validation Balanced Accuracy"] >= best_score - 0.02
].copy()

best_min_impurity_decrease = candidates.sort_values(
    "min_impurity_decrease",
    ascending=False
).iloc[0]["min_impurity_decrease"]

print("Selected min_impurity_decrease:", best_min_impurity_decrease)

base_params["min_impurity_decrease"] = float(best_min_impurity_decrease)
print("Updated parameters:")
print(base_params)

In [ ]:
n_estimators_results = []

for n_estimators in [50, 100, 150, 200, 300, 400, 500]:
    params = base_params.copy()
    params["n_estimators"] = n_estimators

    scores = evaluate_random_forest(params)

    n_estimators_results.append({
        "n_estimators": n_estimators,
        **scores
    })

n_estimators_results = pd.DataFrame(n_estimators_results).round(3)
display(n_estimators_results)

plt.figure(figsize=(7, 4))
plt.plot(
    n_estimators_results["n_estimators"],
    n_estimators_results["Training Accuracy"],
    label="Training Data",
    color="#8ECAE6"
)
plt.plot(
    n_estimators_results["n_estimators"],
    n_estimators_results["Validation Accuracy"],
    label="Validation Data",
    color="#023047"
)
plt.xlabel("n_estimators")
plt.ylabel("Accuracy")
plt.title("n_estimators")
plt.legend()
plt.tight_layout()
plt.savefig("n_estimators.png", dpi=300, bbox_inches="tight")
plt.show()

best_score = n_estimators_results["Validation Balanced Accuracy"].max()

candidates = n_estimators_results[
    n_estimators_results["Validation Balanced Accuracy"] >= best_score - 0.02
].copy()

best_n_estimators = candidates.sort_values("n_estimators").iloc[0]["n_estimators"]

print("Selected n_estimators:", best_n_estimators)

base_params["n_estimators"] = int(best_n_estimators)
print("Final selected parameters:")
print(base_params)

In [ ]:
final_model = RandomForestClassifier(
    **base_params,
    random_state=0,
    class_weight="balanced"
)

final_model.fit(X_train, Y_train)

random_forest = final_model

Y_train_pred = final_model.predict(X_train)
Y_validation_pred = final_model.predict(X_validation)
Y_test_pred = final_model.predict(X_test)

test_report = classification_report(
    Y_test,
    Y_test_pred,
    output_dict=True,
    zero_division=0
)

final_results = pd.DataFrame([{
    "n_estimators": base_params["n_estimators"],
    "max_depth": base_params["max_depth"],
    "min_samples_split": base_params["min_samples_split"],
    "min_impurity_decrease": base_params["min_impurity_decrease"],
    "Training Accuracy": accuracy_score(Y_train, Y_train_pred),
    "Validation Accuracy": accuracy_score(Y_validation, Y_validation_pred),
    "Test Accuracy": accuracy_score(Y_test, Y_test_pred),
    "Test Balanced Accuracy": balanced_accuracy_score(Y_test, Y_test_pred),
    "Test Macro Recall": recall_score(
        Y_test,
        Y_test_pred,
        average="macro",
        zero_division=0
    ),
    "Test Macro F1": f1_score(
        Y_test,
        Y_test_pred,
        average="macro",
        zero_division=0
    ),
    "Test High-Risk Recall": test_report["High"]["recall"] if "High" in test_report else np.nan
}]).round(3)

display(final_results)

In [ ]:
dummy_model = DummyClassifier(strategy="most_frequent", random_state=0)
dummy_model.fit(X_train, Y_train)

base_model = RandomForestClassifier(
    n_estimators=30,
    max_depth=1,
    min_samples_split=2,
    min_impurity_decrease=0.0,
    random_state=0,
    class_weight="balanced"
)
base_model.fit(X_train, Y_train)

tuned_model = final_model

models = {
    "Dummy Baseline": dummy_model,
    "Base Random Forest": base_model,
    "Tuned Random Forest": tuned_model
}

splits = {
    "Training": (X_train, Y_train),
    "Validation": (X_validation, Y_validation),
    "Test": (X_test, Y_test)
}

simple_results = []

for model_name, model in models.items():
    for split_name, (X_split, Y_split) in splits.items():
        Y_pred = model.predict(X_split)

        simple_results.append({
            "Model": model_name,
            "Split": split_name,
            "Accuracy": accuracy_score(Y_split, Y_pred),
            "Precision": precision_score(
                Y_split,
                Y_pred,
                average="macro",
                zero_division=0
            ),
            "Recall": recall_score(
                Y_split,
                Y_pred,
                average="macro",
                zero_division=0
            )
        })

simple_results = pd.DataFrame(simple_results).round(3)

display(simple_results)


In [ ]:
class_distribution = pd.DataFrame({
    "Dataset": ["Full dataset", "Training set", "Validation set", "Test set"],
    "High": [
        int(Y.value_counts().reindex(["High", "Medium", "Low"])["High"]),
        int(Y_train.value_counts().reindex(["High", "Medium", "Low"])["High"]),
        int(Y_validation.value_counts().reindex(["High", "Medium", "Low"])["High"]),
        int(Y_test.value_counts().reindex(["High", "Medium", "Low"])["High"])
    ],
    "Medium": [
        int(Y.value_counts().reindex(["High", "Medium", "Low"])["Medium"]),
        int(Y_train.value_counts().reindex(["High", "Medium", "Low"])["Medium"]),
        int(Y_validation.value_counts().reindex(["High", "Medium", "Low"])["Medium"]),
        int(Y_test.value_counts().reindex(["High", "Medium", "Low"])["Medium"])
    ],
    "Low": [
        int(Y.value_counts().reindex(["High", "Medium", "Low"])["Low"]),
        int(Y_train.value_counts().reindex(["High", "Medium", "Low"])["Low"]),
        int(Y_validation.value_counts().reindex(["High", "Medium", "Low"])["Low"]),
        int(Y_test.value_counts().reindex(["High", "Medium", "Low"])["Low"])
    ],
    "Total": [len(Y), len(Y_train), len(Y_validation), len(Y_test)]
})

table_style = [
    {
        "selector": "caption",
        "props": [
            ("caption-side", "bottom"),
            ("font-weight", "bold"),
            ("font-size", "12px"),
            ("color", "#666666")
        ]
    },
    {
        "selector": "th",
        "props": [
            ("background-color", "#023047"),
            ("color", "white"),
            ("font-weight", "bold"),
            ("text-align", "center"),
            ("border", "1px solid #D9D9D9")
        ]
    },
    {
        "selector": "td",
        "props": [
            ("text-align", "center"),
            ("border", "1px solid #D9D9D9"),
            ("padding", "6px")
        ]
    }
]

display(
    class_distribution.style
    .hide(axis="index")
    .set_table_styles(table_style)
)

feature_pattern_summary = pd.DataFrame({
    "Measure": [
        "Total rows",
        "Unique feature patterns",
        "Duplicate feature patterns"
    ],
    "Value": [
        X.shape[0],
        X.drop_duplicates().shape[0],
        X.duplicated().sum()
    ]
})

display(
    feature_pattern_summary.style
    .hide(axis="index")
    .set_table_styles(table_style)
)


In [ ]:
class_order = ["Low", "Medium", "High"]
cm = confusion_matrix(Y_test, Y_test_pred, labels=class_order)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_order).plot(
    cmap="Blues",
    values_format="d",
    colorbar=False,
    ax=ax
)
ax.set_title("Tuned Random Forest Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
tree_number = 0

plt.figure(figsize=(24, 12))

plot_tree(
    final_model.estimators_[tree_number],
    feature_names=X_train.columns,
    class_names=[str(c) for c in final_model.classes_],
    filled=True,
    rounded=True,
    fontsize=8
)

plt.title("Decision Tree 1 from Final Random Forest")
plt.savefig("decision_tree_1.png", dpi=300, bbox_inches="tight")
plt.show()
tree_number = 49

plt.figure(figsize=(24, 12))

plot_tree(
    final_model.estimators_[tree_number],
    feature_names=X_train.columns,
    class_names=[str(c) for c in final_model.classes_],
    filled=True,
    rounded=True,
    fontsize=8
)

plt.title("Decision Tree 50 from Final Random Forest")
plt.savefig("decision_tree_50.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": final_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

display(feature_importance.round(3))

plt.figure(figsize=(8, 5))

plt.bar(
    feature_importance["Feature"],
    feature_importance["Importance"],
     color="#8ECAE6",
     edgecolor="black"
)

plt.title("Feature Importance of Final Random Forest Model")
plt.xlabel("Feature")
plt.ylabel("Importance Score")

plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()